# LANESHIFT x Kev-0.8B: first LoRA fine-tune

Fine-tune the released `jaredpalmer/kev-0.8b@v1.0` checkpoint with `--init_from` on the 25,427-row `kev-v2-scene` dataset, then compare the original and fine-tuned checkpoints on the same seed-disjoint held-out rows.

**Upload these seven files from `cris@ds/.portable-dev/kev-v2-scene/`:**

- `kev_train.jsonl` - 16,795 rows (128 teacher seeds + 3 human seeds)
- `kev_validation.jsonl` - 4,315 rows, 32 teacher seeds
- `kev_test.jsonl` - 4,317 rows, 32 teacher seeds
- `kev_train.provenance.jsonl`, `kev_validation.provenance.jsonl`, `kev_test.provenance.jsonl`
- `manifest.json`

**Outputs:** `/content/runs/laneshift-kev-v2-scene` (adapter, pointer head, tokenizer, training config), `/content/eval_metrics.json`, `/content/predictions_*.jsonl`, `/content/checkpoint_manifest.json`, `/content/gameplay_eval_plan.json`.

**Claim boundary:** the labels are the deterministic teacher's accepted actions. This run uses the schema-2 scene prompt. This is an imitation experiment, not evidence that Kev drives better. Gameplay improvement requires a matched exported or served checkpoint.


## 1. Runtime

Kev's pinned upstream commit requires Python 3.12 or 3.13. The training and evaluation below need a CUDA GPU. A T4 is enough for LoRA; it does not support bf16, so this notebook uses fp32 there and bf16 on a bf16-capable GPU.

Choose **Runtime > Change runtime type > T4 GPU** (or L4/A100) before running.

In [ ]:
import sys
import torch

print("python", sys.version)
if not ((3, 12) <= sys.version_info[:2] < (3, 14)):
    raise SystemExit("Kev requires Python 3.12 or 3.13. Change the Colab runtime before continuing.")
print("torch", torch.__version__)
if not torch.cuda.is_available():
    raise SystemExit("No CUDA GPU. Runtime > Change runtime type > T4/L4/A100.")
print("gpu", torch.cuda.get_device_name(0))
BF16 = bool(torch.cuda.is_bf16_supported())
print("bf16_supported", BF16)

## 2. Install the pinned upstream Kev source

The handoff does not vendor the training code. This downloads commit `2074cc44256200bb37540103fa4572901faa10f3` and installs it into the notebook kernel.

In [ ]:
import pathlib
import subprocess
import sys

UPSTREAM_SHA = "2074cc44256200bb37540103fa4572901faa10f3"
KEV_SRC = pathlib.Path("/content/kev")
ARCHIVE = pathlib.Path("/content/kev-src.tar.gz")

if not (KEV_SRC / "kev" / "train.py").exists():
    if not ARCHIVE.exists():
        subprocess.run([
            "wget", "-q",
            f"https://codeload.github.com/jaredpalmer/kev/tar.gz/{UPSTREAM_SHA}",
            "-O", str(ARCHIVE),
        ], check=True)
    KEV_SRC.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xzf", str(ARCHIVE), "-C", str(KEV_SRC), "--strip-components=1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(KEV_SRC)], check=True)
print("installed", KEV_SRC)

## 3. Upload the dataset

Upload the seven files listed at the top. The notebook writes them to `/content/data`; if they are already there, it uses them without prompting.

In [ ]:
from google.colab import files
import pathlib

DATA = pathlib.Path("/content/data")
DATA.mkdir(parents=True, exist_ok=True)
REQUIRED = [
    "kev_train.jsonl",
    "kev_validation.jsonl",
    "kev_test.jsonl",
    "kev_train.provenance.jsonl",
    "kev_validation.provenance.jsonl",
    "kev_test.provenance.jsonl",
    "manifest.json",
]

missing = [name for name in REQUIRED if not (DATA / name).exists()]
if missing:
    print("missing:", missing)
    uploaded = files.upload()
    for name, payload in uploaded.items():
        (DATA / name).write_bytes(payload)
missing = [name for name in REQUIRED if not (DATA / name).exists()]
if missing:
    raise SystemExit(f"still missing after upload: {missing}")
for name in REQUIRED:
    print(f"{name:38} {(DATA / name).stat().st_size:>9} bytes")

## 4. Validate the dataset before spending GPU time

These checks mirror the exporter's gates: the counts and action distribution match `manifest.json`, every seed is in exactly one split, every label is one of the offered legal actions, and each record has exactly one `next_action` choice question. The test split is checked here but is **not** used for training.

In [ ]:
import collections
import json
import pathlib

DATA = pathlib.Path("/content/data")
ACTIONS = ("LEFT", "STAY", "RIGHT", "BRAKE")
SPLITS = ("train", "validation", "test")
manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))


def load_jsonl(path):
    return [json.loads(line) for line in pathlib.Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


rows = {split: load_jsonl(DATA / f"kev_{split}.jsonl") for split in SPLITS}
meta = {split: load_jsonl(DATA / f"kev_{split}.provenance.jsonl") for split in SPLITS}
seeds = {}
for split in SPLITS:
    expected = manifest["splits"][split]
    assert len(rows[split]) == expected["rows"], (split, len(rows[split]), expected["rows"])
    assert len(meta[split]) == len(rows[split])
    counts = collections.Counter(row["questions"]["next_action"]["label"] for row in rows[split])
    assert dict(sorted(counts.items())) == expected["action_counts"], (split, counts, expected["action_counts"])
    split_seeds = {item["scenario_seed"] for item in meta[split]}
    seeds[split] = split_seeds
    assert sorted(split_seeds) == expected["seeds"], (split, sorted(split_seeds), expected["seeds"])
    for row, item in zip(rows[split], meta[split]):
        assert item["split"] == split
        assert item["prompt_sha256"], "provenance is missing prompt_sha256"
        assert isinstance(row["state"], str) and row["state"].startswith(("c1\n", "v2\n"))
        assert set(row["questions"]) == {"next_action"}
        question = row["questions"]["next_action"]
        assert question["type"] == "choice"
        assert question["label"] in question["criteria"]
        assert set(question["criteria"]) <= set(ACTIONS)
        assert "STAY" in question["criteria"]
        assert len(question["criteria"]) >= 2
        assert len(row["state"]) <= 2000
    print(f"{split:10} rows={len(rows[split]):4} seeds={sorted(split_seeds)} actions={dict(sorted(counts.items()))}")

for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert not (seeds[left] & seeds[right]), f"seed overlap between {left} and {right}"
print("seed-disjoint: OK")
print("manifest gates:", manifest["quality_gates"])
print("max state chars:", max(len(row["state"]) for split in SPLITS for row in rows[split]))

## 5. Train the LoRA adapter

This is a **delta fine-tune**: `--init_from` warm-starts the LoRA adapter and pointer head from the released checkpoint, then trains on the LANESHIFT training split. The released checkpoint's configuration is preserved: LoRA rank 16, head dimension 256, base `Qwen/Qwen3.5-0.8B-Base` pinned at `dc7cdfe2ee4154fa7e30f5b51ca41bfa40174e68`.

Two choices are deliberate:

- No `none-of-the-above` or distractor augmentation. The four options are the actual legal driving actions, so those synthetic alternatives would teach the wrong label space.
- `--max_state 384` is Kev's minimum training *state-token cap*, not a 384-token model context. The v2 scene rows are at most 256 total tokens. The production ONNX graph is L256. Export and evaluate the original and fine-tuned checkpoints at the same L256 length.

The reference recipe uses three epochs and lr `2e-5` on L4/A100/H100. T4 runs fp32 and two epochs. Keep `--weights_dtype fp32`: the released checkpoint is fp32-weight, and bf16 weights disable merge parity for the fine-tuned side.

In [ ]:
import pathlib
import subprocess
import sys

DATA = pathlib.Path("/content/data")
RUN = pathlib.Path("/content/runs/laneshift-kev-v2-scene")
RUN.parent.mkdir(parents=True, exist_ok=True)

EPOCHS = "3" if BF16 else "2"
TRAIN_DTYPE = "bf16" if BF16 else "fp32"
WEIGHTS_DTYPE = "fp32"  # released checkpoint is fp32-weight; bf16 weights disable merge parity
train_command = [
    sys.executable, "-m", "kev.train",
    "--data", str(DATA / "kev_train.jsonl"),
    "--base", "Qwen/Qwen3.5-0.8B-Base",
    "--base_revision", "dc7cdfe2ee4154fa7e30f5b51ca41bfa40174e68",
    "--init_from", "jaredpalmer/kev-0.8b@v1.0",
    "--lora", "16",
    "--head_dim", "256",
    "--lora_targets", "all",
    "--lr", "2e-5",
    "--epochs", EPOCHS,
    "--batch", "1",
    "--accum", "8",
    "--dtype", TRAIN_DTYPE,
    "--weights_dtype", WEIGHTS_DTYPE,
    "--checkpointing", "1",
    "--max_state", "384",
    "--p_none", "0",
    "--p_none_distract", "0",
    "--p_distract", "0",
    "--p_none_pair", "0",
    "--device", "cuda",
    "--seed", "1",
    "--out", str(RUN),
]
print(" ".join(train_command))
subprocess.run(train_command, check=True)
print("\ncheckpoint files:")
for path in sorted(RUN.iterdir()):
    print(f"  {path.name:32} {path.stat().st_size:>12} bytes")

## 6. Score the original and fine-tuned checkpoints on the same held-out rows

Both models see the exact same records, option sets, and order. Both are scored at temperature 1.0 (raw logits) so the calibration metrics use the same scale; the released checkpoint's fitted serving temperature is recorded separately. Probabilities are masked to the offered legal actions and renormalized to a four-class vector for the standard metrics; the offered-set log loss and Brier are also reported. The original checkpoint is loaded with `--init_from`'s released identity; the fine-tuned model is the local run directory. They are loaded one at a time so a T4 does not hold both.

The NumPy MLP baseline is included as a reference row. It uses a different 33-float input and is not a controlled comparison with Kev; use it only to see the imitation ceiling on this pilot.

In [ ]:
import collections
import gc
import json
import math
import pathlib

import numpy as np
import torch
from kev.checkpoint import LoadOptions
from kev.data import load_records
from kev.predictors import LocalPredictor

DATA = pathlib.Path("/content/data")
RUN = pathlib.Path("/content/runs/laneshift-kev-v2-scene")
ACTIONS = ("LEFT", "STAY", "RIGHT", "BRAKE")
N_CLASSES = len(ACTIONS)
BF16 = globals().get("BF16", torch.cuda.is_bf16_supported())
EVAL_DTYPE = torch.bfloat16 if BF16 else torch.float32


def one_hot(labels):
    out = np.zeros((len(labels), N_CLASSES), dtype=np.float64)
    out[np.arange(len(labels)), np.asarray(labels, dtype=np.int64)] = 1.0
    return out


def confusion_matrix(labels, predictions):
    matrix = np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64)
    for truth, predicted in zip(labels, predictions):
        matrix[int(truth), int(predicted)] += 1
    return matrix


def accuracy(labels, predictions):
    return float(np.mean(np.asarray(labels) == np.asarray(predictions))) if len(labels) else 0.0


def macro_f1(labels, predictions):
    matrix = confusion_matrix(labels, predictions)
    scores = []
    per_class = {}
    for index, name in enumerate(ACTIONS):
        tp = float(matrix[index, index])
        fp = float(matrix[:, index].sum() - tp)
        fn = float(matrix[index, :].sum() - tp)
        support = float(matrix[index, :].sum())
        if support == 0.0:
            continue
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2.0 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class[name] = {"precision": precision, "recall": recall, "f1": f1, "support": int(support)}
        scores.append(f1)
    return (float(np.mean(scores)) if scores else 0.0), per_class


def log_loss(labels, probabilities, epsilon=1e-12):
    if len(labels) == 0:
        return 0.0
    clipped = np.clip(probabilities, epsilon, 1.0)
    return float(-np.mean(np.log(clipped[np.arange(len(labels)), np.asarray(labels, dtype=np.int64)])))


def brier_score(labels, probabilities):
    if len(labels) == 0:
        return 0.0
    return float(np.mean(np.sum((probabilities - one_hot(labels)) ** 2, axis=1)))


def expected_calibration_error(labels, probabilities, bins=10):
    if len(labels) == 0:
        return 0.0, []
    labels = np.asarray(labels, dtype=np.int64)
    confidence = probabilities.max(axis=1)
    predictions = probabilities.argmax(axis=1)
    correct = (predictions == labels).astype(np.float64)
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    table = []
    for index in range(bins):
        low, high = edges[index], edges[index + 1]
        if index == bins - 1:
            mask = (confidence >= low) & (confidence <= high)
        else:
            mask = (confidence >= low) & (confidence < high)
        count = int(mask.sum())
        if count == 0:
            continue
        bin_confidence = float(confidence[mask].mean())
        bin_accuracy = float(correct[mask].mean())
        ece += (count / len(labels)) * abs(bin_accuracy - bin_confidence)
        table.append({"bin": [round(low, 2), round(high, 2)], "count": count,
                      "confidence": round(bin_confidence, 4), "accuracy": round(bin_accuracy, 4)})
    return float(ece), table


def evaluate(labels, probabilities):
    predictions = probabilities.argmax(axis=1)
    macro, per_class = macro_f1(labels, predictions)
    ece, table = expected_calibration_error(labels, probabilities)
    actual = np.bincount(np.asarray(labels, dtype=np.int64), minlength=N_CLASSES)
    predicted = np.bincount(predictions, minlength=N_CLASSES)
    return {
        "rows": int(len(labels)),
        "accuracy": round(accuracy(labels, predictions), 4),
        "macro_f1": round(macro, 4),
        "log_loss": round(log_loss(labels, probabilities), 4),
        "brier": round(brier_score(labels, probabilities), 4),
        "ece": round(ece, 4),
        "mean_confidence": round(float(probabilities.max(axis=1).mean()), 4) if len(labels) else 0.0,
        "per_class": {name: {key: (round(value, 4) if isinstance(value, float) else value) for key, value in stats.items()}
                      for name, stats in per_class.items()},
        "actual_counts": {ACTIONS[i]: int(actual[i]) for i in range(N_CLASSES)},
        "predicted_counts": {ACTIONS[i]: int(predicted[i]) for i in range(N_CLASSES)},
        "confusion_matrix": confusion_matrix(labels, predictions).tolist(),
        "calibration_bins": table,
    }


def score_split(predictor, records, provenance, split):
    labels, probabilities, offered_log_losses, offered_brier = [], [], [], []
    rows = []
    for record, info in zip(records, provenance):
        question = record["questions"]["next_action"]
        label = question["label"]
        offered = list(question["criteria"])
        out = predictor(record)
        probs = out["probabilities"]["next_action"]
        vec = np.array([float(probs.get(action, 0.0)) for action in ACTIONS], dtype=np.float64)
        total = vec.sum()
        if total <= 0:
            raise RuntimeError(f"empty distribution for {info['example_id']}")
        vec /= total
        label_index = ACTIONS.index(label)
        predicted_index = int(vec.argmax())
        labels.append(label_index)
        probabilities.append(vec)
        offered_probabilities = np.array([float(probs[action]) for action in offered], dtype=np.float64)
        offered_probabilities /= offered_probabilities.sum()
        offered_label_index = offered.index(label)
        offered_log_losses.append(float(-math.log(max(offered_probabilities[offered_label_index], 1e-12))))
        offered_brier.append(float(np.sum((offered_probabilities - np.eye(len(offered))[offered_label_index]) ** 2)))
        rows.append({
            "split": split,
            "example_id": info["example_id"],
            "scenario_seed": info["scenario_seed"],
            "episode_id": info["episode_id"],
            "label": label,
            "predicted": ACTIONS[predicted_index],
            "offered": offered,
            "probabilities": {action: round(float(probs[action]), 6) for action in offered},
        })
    metrics = evaluate(labels, np.asarray(probabilities))
    metrics["offered_log_loss"] = round(float(np.mean(offered_log_losses)), 4)
    metrics["offered_brier"] = round(float(np.mean(offered_brier)), 4)
    metrics["per_seed"] = {}
    for seed in sorted({info["scenario_seed"] for info in provenance}):
        indices = [index for index, info in enumerate(provenance) if info["scenario_seed"] == seed]
        seed_metrics = evaluate([labels[index] for index in indices],
                                np.asarray([probabilities[index] for index in indices]))
        seed_metrics["offered_log_loss"] = round(float(np.mean([offered_log_losses[index] for index in indices])), 4)
        seed_metrics["offered_brier"] = round(float(np.mean([offered_brier[index] for index in indices])), 4)
        metrics["per_seed"][str(seed)] = seed_metrics
    return metrics, rows


heldout = {}
for split in ("validation", "test"):
    heldout[split] = (
        load_records(DATA / f"kev_{split}.jsonl"),
        [json.loads(line) for line in (DATA / f"kev_{split}.provenance.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()],
    )


def evaluate_checkpoint(name, run):
    print(f"loading {name}: {run}", flush=True)
    predictor = LocalPredictor(run, "cuda", LoadOptions(dtype=EVAL_DTYPE, merge=True, temperature=1.0))
    result = {}
    for split, (records, provenance) in heldout.items():
        metrics, prediction_rows = score_split(predictor, records, provenance, split)
        result[split] = metrics
        path = pathlib.Path(f"/content/predictions_{name}_{split}.jsonl")
        path.write_text("\n".join(json.dumps(row, sort_keys=True) for row in prediction_rows) + "\n", encoding="utf-8")
        print(f"  {split:10} rows={metrics['rows']:4} accuracy={metrics['accuracy']:.4f} "
              f"macro_f1={metrics['macro_f1']:.4f} log_loss={metrics['log_loss']:.4f} "
              f"brier={metrics['brier']:.4f} ece={metrics['ece']:.4f}")
    result["_temperature"] = predictor.temperature
    del predictor
    gc.collect()
    torch.cuda.empty_cache()
    return result


results = {
    "original": evaluate_checkpoint("original", "jaredpalmer/kev-0.8b@v1.0"),
    "fine_tuned": evaluate_checkpoint("fine_tuned", str(RUN)),
}

## 7. Comparison tables

Accuracy and macro F1 measure imitation. Log loss, Brier, and ECE measure calibration. The rule-based teacher is the target; these numbers are not driving results.

In [ ]:
def print_table(split):
    print(f"\n{split}")
    print(f"{'model':14} {'rows':>5} {'acc':>7} {'macroF1':>8} {'logloss':>8} {'Brier':>7} {'ECE':>7} {'STAYrate':>9}")
    for name in ("original", "fine_tuned"):
        metrics = results[name][split]
        stay_rate = metrics["predicted_counts"]["STAY"] / metrics["rows"] if metrics["rows"] else 0.0
        print(f"{name:14} {metrics['rows']:5d} {metrics['accuracy']:7.4f} {metrics['macro_f1']:8.4f} "
              f"{metrics['log_loss']:8.4f} {metrics['brier']:7.4f} {metrics['ece']:7.4f} {stay_rate:9.3f}")
    reference = {"accuracy": 0.8844, "macro_f1": 0.4426, "log_loss": 0.3587, "brier": 0.1763, "ece": 0.0671}
    if split == "test" and manifest["prompt"]["mode"] == "compact-c1":
        print(f"{'NumPy MLP*':14} {len(rows['test']):5d} {reference['accuracy']:7.4f} {reference['macro_f1']:8.4f} "
              f"{reference['log_loss']:8.4f} {reference['brier']:7.4f} {reference['ece']:7.4f} {'n/a':>9}")
        print("* 33-float MLP from the pilot report; different input, not a controlled Kev comparison.")


print_table("validation")
print_table("test")
print("\nPer-seed test results:")
for name in ("original", "fine_tuned"):
    for seed, metrics in results[name]["test"]["per_seed"].items():
        print(f"  {name:12} seed={seed:>3} rows={metrics['rows']:3d} acc={metrics['accuracy']:.4f} "
              f"macroF1={metrics['macro_f1']:.4f} logloss={metrics['log_loss']:.4f} brier={metrics['brier']:.4f} ece={metrics['ece']:.4f}")

## 8. Save provenance

The checkpoint manifest pins the dataset hash, upstream commit, training command, adapter/head hashes, and the evaluator settings. Keep it with the first checkpoint so later comparisons can be reproduced.

In [ ]:
import hashlib
import json
import pathlib

if "train_command" not in globals():
    train_command = ["see notebooks/KEV-V2-SCENE.md; run train_gpu.sh, skip the training cell"]
UPSTREAM_SHA = globals().get("UPSTREAM_SHA", "2074cc44256200bb37540103fa4572901faa10f3")

RUN = pathlib.Path("/content/runs/laneshift-kev-v2-scene")
DATA = pathlib.Path("/content/data")


def sha256(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
checkpoint = {
    "run": str(RUN),
    "files": {
        name: {"bytes": (RUN / name).stat().st_size, "sha256": sha256(RUN / name)}
        for name in ("adapter_config.json", "adapter_model.safetensors", "head.pt", "training_config.json")
        if (RUN / name).exists()
    },
    "training_command": train_command,
    "dataset": {
        "manifest_sha256": sha256(DATA / "manifest.json"),
        "approval_file_sha256": manifest["approval_file_sha256"],
        "splits": {split: manifest["splits"][split]["rows"] for split in ("train", "validation", "test")},
        "prompt_mode": manifest["prompt"]["mode"],
    },
    "evaluation": {
        "dtype": str(EVAL_DTYPE).replace("torch.", ""),
        "temperature": 1.0,
        "checkpoint_temperatures": {name: results[name].get("_temperature") for name in results},
        "upstream_sha": UPSTREAM_SHA,
        "heldout_rows": {split: manifest["splits"][split]["rows"] for split in ("validation", "test")},
    },
    "claim_boundary": "teacher-labelled imitation with the manifest's prompt mode; offline metrics only, no gameplay claim",
}
pathlib.Path("/content/checkpoint_manifest.json").write_text(json.dumps(checkpoint, indent=2, sort_keys=True) + "\n", encoding="utf-8")
pathlib.Path("/content/eval_metrics.json").write_text(json.dumps(results, indent=2, sort_keys=True) + "\n", encoding="utf-8")
pathlib.Path("/content/gameplay_eval_plan.json").write_text(json.dumps({
    "checkpoint": checkpoint,
    "held_out_gameplay_seeds": manifest["splits"]["test"]["seeds"],
    "conditions": {"episode_duration_s": 30.0, "traffic_density": 1.5, "sim_speed_scale": 2.0},
    "policies": ["rule_based", "original_kev", "fine_tuned_kev"],
    "note": "Run every policy on the same seed list. The fine-tuned adapter must first be merged/exported to ONNX or served; this notebook does not run the game.",
}, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(checkpoint, indent=2, sort_keys=True))

## 9. Keep the checkpoint off Colab

This copies the run directory and the small result files to Google Drive. It does **not** create a ZIP. If Drive is unavailable, download `adapter_model.safetensors`, `adapter_config.json`, `head.pt`, and the tokenizer files individually.

In [ ]:
import pathlib
import shutil

from google.colab import drive

drive.mount("/content/drive")
destination = pathlib.Path("/content/drive/MyDrive/LANESHIFT/kev-lora-pilot")
destination.mkdir(parents=True, exist_ok=True)
shutil.copytree("/content/runs/laneshift-kev-v2-scene", destination / "checkpoint", dirs_exist_ok=True)
for name in ("eval_metrics.json", "checkpoint_manifest.json", "gameplay_eval_plan.json"):
    shutil.copy2(f"/content/{name}", destination / name)
print("saved", destination)

## 10. Next: gameplay comparison

The offline comparison above is the first checkpoint's decision-level evidence. The gameplay comparison needs a checkpoint the LANESHIFT controller can load. The current local path is ONNX; the source-only alternative is a Kev System One HTTP server.

1. Merge the LoRA adapter into the base and export an ONNX graph for the fine-tuned checkpoint, labeled `fine_tuned_model`. The original graph export source is in `tools/kev-onnx-export/`; it currently targets the released model and must be pointed at the merged fine-tuned weights.
2. Run the existing matched experiment runner on the held-out seeds with the **same** duration, traffic density, simulation speed, decision timeout, and staleness bounds for the original and fine-tuned ONNX models:
   `python python/experiment_runner.py --policies rule_based,onnx_kev --seeds 7,29,61 --duration 30 --traffic-density 1.5 --sim-speed-scale 2.0 --onnx-model <model.onnx> ... --onnx-model-id <id> --output <run>.csv`.
3. Report score, survival, collisions, action distribution, decision count, and latency for each policy. Keep inference latency as its own measurement; do not gate the training path on QNN, Adreno, or NPU work.

The immediate milestone is the verified host workspace, the seed-disjoint dataset, this notebook, and the first fine-tuned checkpoint. The first LoRA run is complete when `eval_metrics.json` exists and the adapter/head hashes are recorded.